# 04 — Memória Conversacional

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 05_Agentes  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Memória de curto prazo** — histórico completo da conversa (janela deslizante)
- **Memória de longo prazo** — fatos persistidos entre sessões em arquivo JSON
- **Sumarização automática** — quando o contexto enche, o LLM comprime o histórico
- **Memória semântica** — busca por relevância em vez de manter tudo

---

### Por que gerenciar memória?

LLMs têm uma janela de contexto limitada. Em conversas longas, duas coisas acontecem:

| Problema | Consequência |
|---|---|
| Contexto excede o limite | Erro da API ou truncamento silencioso |
| Contexto muito longo | Custo alto + resposta mais lenta |
| Sessão encerrada | Tudo que foi dito é perdido |

```
Memória curto prazo  → mantém N últimas mensagens (janela deslizante)
Memória longo prazo  → persiste fatos importantes em disco
Sumarização          → comprime histórico antigo quando contexto enche
Memória semântica    → busca mensagens relevantes por similaridade
```

## Setup

In [9]:
import sys, os, json, time
from pathlib import Path
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL = os.getenv('LLM_MODEL', 'deepseek-chat')
DATA_DIR  = Path('../data/memoria')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM     : {LLM_MODEL}')
print(f'Memória : {DATA_DIR.resolve()}')

LLM     : deepseek-chat
Memória : C:\Users\Jorge Maques\Documents\Especialista_em_AI\EAI_07_AI_Generative\data\memoria


---
## 1. Memória de Curto Prazo — Janela Deslizante

A estratégia mais simples: manter apenas as **N últimas mensagens**.  
Quando o histórico excede o limite, as mensagens mais antigas são descartadas.

Vantagem: simples, sem custo extra.  
Desvantagem: perde contexto antigo — o agente "esquece" o início da conversa.

In [10]:
agente = AgenteCurtoPrazo(
    system='Você é um assistente técnico do curso Especialista em IA.',
    max_mensagens=4
)

conversa = [
    'Meu nome é Carlos e estou estudando o módulo EAI_07.',
    'O que é RAG?',
    'E o que é BM25?',
    'Qual foi o primeiro assunto que mencionei nessa conversa?',
]

print('=== Janela deslizante (max=4 msgs) ===\n')
for msg in conversa:
    print(f'👤 {msg}')
    resp = agente.responder(msg, verbose=True)
    print(f'🤖 {resp}\n')


=== Janela deslizante (max=4 msgs) ===

👤 Meu nome é Carlos e estou estudando o módulo EAI_07.
  [memória] 1 msgs total | 1 no contexto
🤖 Olá, Carlos! É um prazer recebê-lo aqui. 😊

O módulo **EAI_07** é uma parte importante da sua jornada no curso de Especialista em IA. Para que eu possa ajudá-lo da melhor forma possível, você poderia me contar um pouco mais sobre:

1. **Qual é o tema específico** do módulo EAI_07 que você está estudando?
2. **Em que parte** você está tendo dúvidas ou gostaria de aprofundar?
3. Há algum **conceito, exercício ou projeto** em particular que gostaria de discutir?

Estou aqui para auxiliá-lo com explicações, exemplos, resumos ou qualquer outro suporte que precise para dominar o conteúdo. Pode compartilhar suas dúvidas à vontade! 🚀

👤 O que é RAG?
  [memória] 3 msgs total | 3 no contexto
🤖 **RAG (Retrieval-Augmented Generation)** é uma arquitetura avançada de IA que combina **recuperação de informações** com **geração de texto**, permitindo que modelos de 

---
## 2. Memória de Longo Prazo — Persistência em Disco

Fatos importantes são extraídos da conversa e salvos em JSON.  
Na próxima sessão, esses fatos são injetados no system prompt.

O LLM decide quais fatos merecem ser lembrados — nome do usuário,
preferências, contexto do projeto, decisões tomadas.

In [11]:
MEMORIA_PATH = DATA_DIR / 'memoria_longo_prazo.json'


def carregar_memoria() -> list:
    """Carrega fatos persistidos de sessões anteriores."""
    if MEMORIA_PATH.exists():
        with open(MEMORIA_PATH, encoding='utf-8') as f:
            return json.load(f)
    return []


def salvar_memoria(fatos: list):
    """Persiste fatos em disco."""
    with open(MEMORIA_PATH, 'w', encoding='utf-8') as f:
        json.dump(fatos, f, ensure_ascii=False, indent=2)


def extrair_fatos(historico: list) -> list:
    """
    Usa o LLM para extrair fatos relevantes da conversa.
    Retorna lista de strings com os fatos identificados.
    """
    if not historico:
        return []

    conversa_texto = '\n'.join(
        f'{m["role"].upper()}: {m["content"]}'
        for m in historico
    )
    prompt = f"""Analise esta conversa e extraia fatos importantes sobre o usuário
que devem ser lembrados em conversas futuras.

Extraia apenas:
- Nome, preferências, contexto pessoal
- Decisões técnicas tomadas
- Projetos ou módulos em andamento
- Problemas resolvidos ou pendentes

Conversa:
{conversa_texto}

Responda APENAS com JSON válido:
{{"fatos": ["fato 1", "fato 2", ...]}}

Se não houver fatos relevantes, retorne {{"fatos": []}}"""

    resp = llm.chat.completions.create(
        model       = LLM_MODEL,
        messages    = [{'role': 'user', 'content': prompt}],
        temperature = 0.0,
    )
    texto = resp.choices[0].message.content.strip()
    texto = __import__('re').sub(r'^```json\s*|^```\s*|\s*```$', '', texto, flags=8).strip()
    try:
        return json.loads(texto).get('fatos', [])
    except json.JSONDecodeError:
        return []


class AgenteLongoPrazo:
    """
    Agente com memória persistente entre sessões.
    Extrai fatos importantes ao final de cada conversa e os carrega na próxima.
    """

    def __init__(self, system_base: str, max_mensagens: int = 20):
        self.system_base   = system_base
        self.max_mensagens = max_mensagens
        self.historico     = []
        self.fatos         = carregar_memoria()
        if self.fatos:
            print(f'Memória carregada: {len(self.fatos)} fato(s)')
            for f in self.fatos:
                print(f'  • {f}')

    def _system_com_memoria(self) -> str:
        """Injeta os fatos persistidos no system prompt."""
        if not self.fatos:
            return self.system_base
        fatos_texto = '\n'.join(f'- {f}' for f in self.fatos)
        return f"{self.system_base}\n\nO que você sabe sobre o usuário:\n{fatos_texto}"

    def responder(self, mensagem: str) -> str:
        self.historico.append({'role': 'user', 'content': mensagem})
        janela = self.historico[-self.max_mensagens:]
        resp = llm.chat.completions.create(
            model    = LLM_MODEL,
            messages = [{'role': 'system', 'content': self._system_com_memoria()}] + janela,
        )
        resposta = resp.choices[0].message.content
        self.historico.append({'role': 'assistant', 'content': resposta})
        return resposta

    def encerrar_sessao(self):
        """Extrai fatos da conversa atual e persiste em disco."""
        novos = extrair_fatos(self.historico)
        # Merge: evita duplicatas
        todos = list({f for f in self.fatos + novos})
        salvar_memoria(todos)
        print(f'Sessão encerrada. {len(novos)} novo(s) fato(s) salvo(s):')
        for f in novos:
            print(f'  • {f}')
        self.fatos     = todos
        self.historico = []

    def limpar_memoria(self):
        """Apaga todos os fatos persistidos."""
        self.fatos = []
        if MEMORIA_PATH.exists():
            MEMORIA_PATH.unlink()
        print('Memória de longo prazo apagada.')


print('AgenteLongoPrazo definido.')

AgenteLongoPrazo definido.


In [12]:
agente_lp = AgenteLongoPrazo(
    system_base='Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.'
)

print('=== SESSÃO 1 ===\n')
trocas = [
    'Oi! Me chamo Carlos, estou no módulo EAI_07 e uso DeepSeek como provider.',
    'Estou tendo dificuldade com o formato DSML do DeepSeek no function calling.',
    'Resolvi usando o tool_runner.py do shared/. Funcionou bem.',
]
for msg in trocas:
    print(f'👤 {msg}')
    print(f'🤖 {agente_lp.responder(msg)}\n')

agente_lp.encerrar_sessao()


=== SESSÃO 1 ===

👤 Oi! Me chamo Carlos, estou no módulo EAI_07 e uso DeepSeek como provider.
🤖 Olá Carlos! Que bom te ver por aqui! 👋 

Como aluno do módulo EAI_07 usando DeepSeek como provider, você está no caminho certo para dominar técnicas avançadas de IA. O DeepSeek é uma excelente escolha - potente, versátil e com ótima relação custo-benefício.

Como posso te ajudar hoje? Tem alguma dúvida específica sobre:
- Implementação de modelos com DeepSeek?
- Conceitos do módulo EAI_07?
- Otimização de prompts ou arquitetura?
- Projetos práticos que está desenvolvendo?

Estou aqui para te auxiliar no que precisar! 😊

👤 Estou tendo dificuldade com o formato DSML do DeepSeek no function calling.
🤖 Entendo perfeitamente, Carlos! O formato DSML (DeepSeek Markup Language) do DeepSeek pode ser um pouco confuso no início, especialmente para function calling. Vou te explicar de forma clara:

## **Formato DSML para Function Calling**

### **Estrutura Básica:**
```xml
<｜begin▁of▁sentence｜>
<｜begin▁

In [13]:
agente_lp2 = AgenteLongoPrazo(
    system_base='Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.'
)

print('\n=== SESSÃO 2 (nova instância) ===\n')
trocas2 = [
    'Olá! Lembra de mim?',
    'Qual provider estou usando no meu projeto?',
    'Como resolvi o problema de function calling que tive?',
]
for msg in trocas2:
    print(f'👤 {msg}')
    print(f'🤖 {agente_lp2.responder(msg)}\n')

agente_lp2.limpar_memoria()


Memória carregada: 6 fato(s)
  • Usa DeepSeek como provider
  • Nome: Carlos
  • Resolveu o problema usando o tool_runner.py da pasta shared/
  • Teve dificuldade com o formato DSML do DeepSeek para function calling
  • Está no módulo EAI_07
  • Demonstrou capacidade de resolver problemas técnicos de forma prática

=== SESSÃO 2 (nova instância) ===

👤 Olá! Lembra de mim?
🤖 Olá Carlos! Claro que me lembro! 😊

Você é o aluno do curso Especialista em IA que estava trabalhando com o módulo EAI_07, usando o DeepSeek como provider e teve aquela questão com o formato DSML para function calling. Lembro que você resolveu o problema de forma prática usando o `tool_runner.py` da pasta shared/.

Como está indo com os estudos? Conseguiu avançar mais no módulo ou tem alguma nova dúvida sobre function calling, DSML ou qualquer outro aspecto do curso? Estou aqui para ajudar!

👤 Qual provider estou usando no meu projeto?
🤖 Você está usando o **DeepSeek** como provider no seu projeto! 🎯

Lembro perfeita

---
## 3. Sumarização Automática

Quando o histórico excede um limite de tokens estimado, o LLM comprime
as mensagens antigas em um resumo — que substitui o histórico completo.

Vantagem: mantém contexto semântico sem crescimento ilimitado.  
Desvantagem: custo extra de uma chamada de sumarização.

In [14]:
def estimar_tokens(mensagens: list) -> int:
    """Estimativa rápida: ~4 caracteres por token."""
    total = sum(len(m['content']) for m in mensagens)
    return total // 4


def sumarizar_historico(historico: list) -> str:
    """
    Comprime o histórico em um resumo conciso.
    Mantém fatos, decisões e contexto importantes.
    """
    texto = '\n'.join(
        f'{m["role"].upper()}: {m["content"]}'
        for m in historico
    )
    resp = llm.chat.completions.create(
        model    = LLM_MODEL,
        messages = [
            {'role': 'system', 'content':
                'Você resume conversas técnicas. Preserve: fatos sobre o usuário, '
                'decisões técnicas, problemas e soluções. Seja conciso.'
            },
            {'role': 'user', 'content': f'Resuma esta conversa em até 200 palavras:\n\n{texto}'},
        ],
        temperature = 0.0,
    )
    return resp.choices[0].message.content


class AgenteComSumarizacao:
    """
    Agente que sumariza o histórico automaticamente quando o contexto fica grande.
    Mantém um resumo comprimido + as N mensagens mais recentes.
    """

    def __init__(
        self,
        system        : str,
        limite_tokens : int = 2000,   # threshold para disparar sumarização
        manter_recentes: int = 4,     # mensagens recentes preservadas após resumo
    ):
        self.system          = system
        self.limite_tokens   = limite_tokens
        self.manter_recentes = manter_recentes
        self.historico       = []
        self.resumo          = ''      # resumo comprimido do histórico antigo
        self.sumarizacoes    = 0       # contador de sumarizações

    def _system_com_resumo(self) -> str:
        if not self.resumo:
            return self.system
        return f"{self.system}\n\nResumo da conversa anterior:\n{self.resumo}"

    def _verificar_e_sumarizar(self):
        """Sumariza o histórico antigo se o contexto exceder o limite."""
        tokens_estimados = estimar_tokens(self.historico)
        if tokens_estimados > self.limite_tokens:
            # Separa: parte a sumarizar + mensagens recentes a preservar
            a_sumarizar    = self.historico[:-self.manter_recentes]
            recentes       = self.historico[-self.manter_recentes:]
            novo_resumo    = sumarizar_historico(a_sumarizar)
            # Acumula resumos
            if self.resumo:
                self.resumo = f"{self.resumo}\n\n{novo_resumo}"
            else:
                self.resumo = novo_resumo
            self.historico    = recentes
            self.sumarizacoes += 1
            print(f'  [sumarização #{self.sumarizacoes}] '
                  f'{tokens_estimados} tokens → resumo + {len(recentes)} msgs recentes')

    def responder(self, mensagem: str, verbose: bool = False) -> str:
        self.historico.append({'role': 'user', 'content': mensagem})
        self._verificar_e_sumarizar()

        if verbose:
            tokens = estimar_tokens(self.historico)
            print(f'  [contexto] ~{tokens} tokens | {len(self.historico)} msgs | '
                  f'{self.sumarizacoes} sumarizações')

        resp = llm.chat.completions.create(
            model    = LLM_MODEL,
            messages = [{'role': 'system', 'content': self._system_com_resumo()}] + self.historico,
        )
        resposta = resp.choices[0].message.content
        self.historico.append({'role': 'assistant', 'content': resposta})
        return resposta


print('AgenteComSumarizacao definido.')

AgenteComSumarizacao definido.


In [15]:
agente_sum = AgenteComSumarizacao(
    system         = 'Você é o Assistente Técnico do curso Especialista em IA.',
    limite_tokens  = 500,
    manter_recentes= 2,
)

topicos = [
    'Estou no módulo EAI_07 estudando agentes. O que é o padrão ReAct?',
    'Que ferramentas customizadas podemos dar a um agente?',
    'Como funciona o pipeline multi-agente com roteador?',
    'O que é sumarização de contexto e quando usar?',
    'Qual a diferença entre memória de curto e longo prazo em agentes?',
]

print('=== Conversa com sumarização automática ===\n')
for msg in topicos:
    print(f'👤 {msg}')
    resp = agente_sum.responder(msg, verbose=True)
    print(f'🤖 {resp}\n')

print(f'\nTotal de sumarizações: {agente_sum.sumarizacoes}')
if agente_sum.resumo:
    print(f'\nResumo acumulado:\n{agente_sum.resumo}')


=== Conversa com sumarização automática ===

👤 Estou no módulo EAI_07 estudando agentes. O que é o padrão ReAct?
  [contexto] ~16 tokens | 1 msgs | 0 sumarizações
🤖 O **padrão ReAct (Reasoning + Acting)** é uma estrutura fundamental no desenvolvimento de agentes de IA, que combina **raciocínio** (Reasoning) e **ação** (Acting) de forma intercalada para resolver tarefas complexas.  

No contexto do módulo EAI_07, ele representa um avanço em relação a abordagens puramente reativas ou baseadas apenas em planejamento, permitindo que o agente:

1. **Pense passo a passo** (Reasoning) – reflete sobre o que sabe, o que precisa fazer e como proceder.  
2. **Aja com base nesse raciocínio** (Acting) – executa ações como consultar uma ferramenta, buscar informações ou interagir com um ambiente.  
3. **Observe os resultados** e continue ciclicamente até concluir a tarefa.

---

### Elementos principais do ReAct:

- **Pensamento (Thought)**: O agente gera uma cadeia de raciocínio interna antes de ag

---
## 4. Comparativo: As Três Estratégias

Mesma conversa longa nas três abordagens — comparamos o que cada uma lembra.

In [16]:
SYSTEM = 'Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.'

agente_cp   = AgenteCurtoPrazo(system=SYSTEM, max_mensagens=4)
agente_sum2 = AgenteComSumarizacao(system=SYSTEM, limite_tokens=400, manter_recentes=2)

contexto = [
    'Sou Carlos, estudo IA e meu provider favorito é DeepSeek.',
    'Concluí os módulos EAI_01 a EAI_06 com sucesso.',
    'No EAI_07, implementei RAG com FAISS e BM25.',
    'Agora estou no submódulo de agentes — já fiz ReAct e multi-agentes.',
]

print('Alimentando histórico com contexto...\n')
for msg in contexto:
    agente_cp.responder(msg)
    agente_sum2.responder(msg)
    print(f'  ✓ "{msg[:60]}"')

pergunta_teste = 'Quais módulos eu já concluí e qual é o meu provider de LLM?'
print(f'\n👤 {pergunta_teste}\n')

print('─── Curto prazo (janela=4) ─────────────────────────')
print(agente_cp.responder(pergunta_teste))

print('\n─── Com sumarização ───────────────────────────────')
print(agente_sum2.responder(pergunta_teste))


Alimentando histórico com contexto...

  ✓ "Sou Carlos, estudo IA e meu provider favorito é DeepSeek."
  ✓ "Concluí os módulos EAI_01 a EAI_06 com sucesso."
  [sumarização #1] 450 tokens → resumo + 2 msgs recentes
  ✓ "No EAI_07, implementei RAG com FAISS e BM25."
  [sumarização #2] 751 tokens → resumo + 2 msgs recentes
  ✓ "Agora estou no submódulo de agentes — já fiz ReAct e multi-a"

👤 Quais módulos eu já concluí e qual é o meu provider de LLM?

─── Curto prazo (janela=4) ─────────────────────────
Com base no nosso histórico, aqui está seu **progresso no curso Especialista em IA**:

## **📚 Módulos Concluídos/Em Andamento:**

1. **EAI_07 - RAG Avançado** ✅
   - Implementou FAISS + BM25
   - Trabalhou com retrieval híbrido

2. **Submódulo de Agentes** (em andamento) 🔄
   - ✅ ReAct (Reasoning + Acting)
   - ✅ Multi-Agentes
   - Próximos: Planejamento, Swarms, etc.

## **🤖 Provider de LLM:**
Pelas nossas conversas anteriores, você está usando **DeepSeek** como seu principal provider! 



---
## Resumo

| Estratégia | Como funciona | Vantagem | Limitação |
|---|---|---|---|
| **Janela deslizante** | Mantém N últimas msgs | Simples, sem custo extra | Perde contexto antigo |
| **Longo prazo** | Extrai e persiste fatos em JSON | Lembra entre sessões | Só fatos explícitos |
| **Sumarização** | Comprime histórico antigo com LLM | Preserva semântica | Custo extra de chamada |

### Quando usar cada uma?

```
Conversa curta / sessão única     → Janela deslizante (max=20)
Assistente pessoal multi-sessão   → Longo prazo (fatos persistidos)
Conversa técnica longa e densa    → Sumarização automática
Produção com usuários reais       → Longo prazo + Sumarização combinados
```

### Parâmetros importantes

```python
AgenteCurtoPrazo(max_mensagens=10)          # 5 turns de ida e volta
AgenteLongoPrazo(max_mensagens=20)          # janela + persistência
AgenteComSumarizacao(
    limite_tokens=2000,                      # dispara ao estimar 2k tokens
    manter_recentes=4,                       # preserva 2 turns recentes
)
```